<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets nltk -q

In [ ]:
from datasets import load_dataset

dataset_lm = load_dataset("wikitext", "wikitext-2-raw-v1")

print(dataset_lm)
print(dataset_lm["train"][10]["text"])

In [ ]:
import re
from collections import Counter, defaultdict
import math
import random

def tokenize_lm(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9.,!?;:'\"()\- ]", " ", text)
    text = re.sub(r"([.,!?;:()])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

def preprocess_lines(split):
    tokenized_sentences = []

    for item in dataset_lm[split]:
        text = item["text"].strip()

        if len(text) == 0:
            continue

        tokens = tokenize_lm(text)

        if len(tokens) < 3:
            continue

        tokenized_sentences.append(["<s>", "<s>"] + tokens + ["</s>"])

    return tokenized_sentences

train_sents = preprocess_lines("train")
val_sents = preprocess_lines("validation")
test_sents = preprocess_lines("test")

print(len(train_sents), len(val_sents), len(test_sents))
print(train_sents[0][:30])

In [ ]:
UNK = "<unk>"

word_counts = Counter()

for sent in train_sents:
    word_counts.update(sent)

min_freq = 2

vocab = {word for word, count in word_counts.items() if count >= min_freq}
vocab.add(UNK)
vocab.add("<s>")
vocab.add("</s>")

print("Vocabulary size:", len(vocab))

In [ ]:
def replace_unk(sent):
    return [word if word in vocab else UNK for word in sent]

train_sents = [replace_unk(sent) for sent in train_sents]
val_sents = [replace_unk(sent) for sent in val_sents]
test_sents = [replace_unk(sent) for sent in test_sents]

In [ ]:
trigram_counts = defaultdict(Counter)
bigram_counts = Counter()

for sent in train_sents:
    for i in range(2, len(sent)):
        context = (sent[i-2], sent[i-1])
        word = sent[i]

        trigram_counts[context][word] += 1
        bigram_counts[context] += 1

print("Number of contexts:", len(trigram_counts))

In [ ]:
V = len(vocab)

def trigram_prob(context, word):
    return (trigram_counts[context][word] + 1) / (bigram_counts[context] + V)

In [ ]:
def perplexity(sentences):
    log_prob_sum = 0
    token_count = 0

    for sent in sentences:
        for i in range(2, len(sent)):
            context = (sent[i-2], sent[i-1])
            word = sent[i]

            prob = trigram_prob(context, word)

            log_prob_sum += math.log(prob)
            token_count += 1

    avg_neg_log_prob = -log_prob_sum / token_count
    return math.exp(avg_neg_log_prob)

val_ppl = perplexity(val_sents)
test_ppl = perplexity(test_sents)

print("Validation Perplexity:", val_ppl)
print("Test Perplexity:", test_ppl)

In [ ]:
def generate_text(max_len=30):
    context = ("<s>", "<s>")
    generated = []

    for _ in range(max_len):
        candidates = list(trigram_counts[context].keys())

        if len(candidates) == 0:
            break

        weights = [trigram_counts[context][w] for w in candidates]
        next_word = random.choices(candidates, weights=weights, k=1)[0]

        if next_word == "</s>":
            break

        generated.append(next_word)
        context = (context[1], next_word)

    return " ".join(generated)

for i in range(5):
    print(f"Sample {i+1}:")
    print(generate_text(max_len=30))
    print()